In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task1')

RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task1/data/IT.csv')
oot = pl.read_csv('task1/data/OOT.csv')
print(f'IT: {it.shape}, OOT: {oot.shape}')
print(f'Target rate: {it["TARGET"].mean():.4f}')

In [ ]:
num_cols = [c for c in it.columns if c.startswith('NUMERICAL_')]
print(f'Numerical features: {len(num_cols)}')

it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
print(f'Time cutoff (80th pctl): {cutoff}')

train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')
print(f'Train target rate: {train["TARGET"].mean():.4f}')
print(f'Val target rate: {val["TARGET"].mean():.4f}')

In [ ]:
X_train = train.select(num_cols).to_pandas().values
y_train = train['TARGET'].to_numpy()
X_val = val.select(num_cols).to_pandas().values
y_val = val['TARGET'].to_numpy()
X_oot = oot.select(num_cols).to_pandas().values

In [ ]:
bp = BinningProcess(
    variable_names=num_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train, y_train)

selected = bp.get_support(names=True)
print(f'Selected features (IV >= 0.02): {len(selected)}')
print(selected)

In [ ]:
summary = bp.summary()
summary = summary.sort_values('iv', ascending=False)
print(summary[['name', 'iv', 'n_bins', 'quality_score']].to_string())

In [ ]:
X_train_woe = bp.transform(X_train, metric='woe')
X_val_woe = bp.transform(X_val, metric='woe')
X_oot_woe = bp.transform(X_oot, metric='woe')
print(f'WOE shape: {X_train_woe.shape}')

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr.fit(X_train_woe, y_train)

p_train = lr.predict_proba(X_train_woe)[:, 1]
p_val = lr.predict_proba(X_val_woe)[:, 1]

train_gini = 2 * roc_auc_score(y_train, p_train) - 1
val_gini = 2 * roc_auc_score(y_val, p_val) - 1
print(f'train_gini: {train_gini:.4f}')
print(f'val_gini: {val_gini:.4f}')

In [ ]:
with mlflow.start_run(run_name='v1_woe_lr_baseline'):
    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('C', 1.0)
    mlflow.log_param('n_features', len(selected))
    mlflow.log_param('features', list(selected))
    mlflow.log_param('min_iv', 0.02)
    mlflow.log_metric('train_gini', train_gini)
    mlflow.log_metric('val_gini', val_gini)
    mlflow.set_tag('task', 'task1')
    mlflow.set_tag('run_tag', RUN_TAG)

In [ ]:
p_oot = lr.predict_proba(X_oot_woe)[:, 1]
preds = pl.DataFrame({
    'ID_APPLICATION': oot['ID_APPLICATION'],
    'SCORE': p_oot
})
preds.write_csv('task1/predictions.csv')
print(f'OOT predictions: {preds.shape}')
print(preds.head())